# Ion Channel vs. Model Quality Analysis

Reads `allen_classified.csv` from `main_results/allen_posterior_mass_analysis/` 
(produced by `analysis/posterior_mass_analysis/02_classify_particles.ipynb`).

**Metric:** `neg_nle = −log_weight = neg_log_marginal_NLE` — lower = better fit.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

from modelsmc.utils.plot_utils import use_style

FIGURES_DIR = Path("../fig")
FIGURES_DIR.mkdir(exist_ok=True)

CLASSIFIED_CSV = Path(
    "../../../main_results/allen_posterior_mass_analysis/allen_classified.csv"
)
assert CLASSIFIED_CSV.exists(), "Run 02_classify_particles.ipynb first."
print(f"Figures → {FIGURES_DIR.resolve()}")

In [ ]:
df = pd.read_csv(CLASSIFIED_CSV)
print(f"Loaded {len(df)} classified particles")
print(f"Seeds: {sorted(df['seed'].unique())}")

df["channel_type"] = df["subtype_id"].fillna("unknown").str.strip()

# neg_nle = neg_log_marginal_NLE = -log_weight  (positive; lower = better fit)
df["neg_nle"] = -df["log_weight"]
print(
    f"log_weight range : [{df['log_weight'].min():.1f}, {df['log_weight'].max():.1f}]"
)
print(f"neg_nle   range  : [{df['neg_nle'].min():.1f}, {df['neg_nle'].max():.1f}]")

# ── Outlier filter: upper Tukey fence on neg_nle ──────────────────────────────
q1, q3 = df["neg_nle"].quantile([0.25, 0.75])
upper_fence = q3 + 3.0 * (q3 - q1)
n_out = (df["neg_nle"] > upper_fence).sum()
print(f"Outlier fence (Q3 + 3·IQR): {upper_fence:.1f}  →  {n_out} particles removed")

df_exp = df[df["neg_nle"] <= upper_fence].copy().reset_index(drop=True)
df_exp = df_exp[df_exp["channel_type"] != "unknown"].copy().reset_index(drop=True)
print(f"df_exp: {len(df_exp)} particles after outlier removal and unknown filter")
print("\nChannel type counts:")
print(df_exp["channel_type"].value_counts().to_string())

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────────────

import matplotlib.cm as _cm

_FAMILY_ORDER = [
    "I_M",
    "base_only",
    "I_M_plus_I_A",
    "I_M_plus_I_h",
    "I_M_plus_I_NaP",
    "I_M_plus_I_Ks",
    "I_M_plus_I_bgK",
]

_FAMILY_CMAP = {
    "base_only": _cm.Greys,
    "I_M": _cm.Oranges,
    "I_M_plus_I_NaP": _cm.Greens,
    "I_M_plus_I_h": _cm.Purples,
    "I_M_plus_I_A": _cm.Blues,
    "I_M_plus_I_Ks": _cm.Reds,
    "I_M_plus_I_bgK": _cm.YlOrBr,
}

# ── Human-readable x-axis labels ─────────────────────────────────────────────
CHANNEL_LABELS = {
    "base_only": "Base",
    "I_M_fixed_tau": r"$I_M$ fixed $\tau$",
    "I_M_voltage_dep_tau": r"$I_M$ V-dep $\tau$",
    "I_M_plus_I_NaP_fixed_tau": r"$I_M$+$I_{NaP}$ fixed $\tau$",
    "I_M_plus_I_NaP_voltage_dep_tau": r"$I_M$+$I_{NaP}$ V-dep $\tau$",
    "I_M_plus_I_h_fixed_tau": r"$I_M$+$I_h$ fixed $\tau$",
    "I_M_plus_I_h_voltage_dep_tau": r"$I_M$+$I_h$ V-dep $\tau$",
    "I_M_plus_I_A": r"$I_M$+$I_A$",
    "I_M_plus_I_Ks": r"$I_M$+$I_{Ks}$",
    "I_M_plus_I_bgK": r"$I_M$+$I_{bgK}$",
}


def _family(ct: str) -> str:
    """Extract ion-channel family from a subtype label."""
    if ct == "base_only":
        return "base_only"
    for fam in [
        "I_M_plus_I_NaP",
        "I_M_plus_I_h",
        "I_M_plus_I_A",
        "I_M_plus_I_Ks",
        "I_M_plus_I_bgK",
    ]:
        if ct.startswith(fam):
            return fam
    if ct.startswith("I_M"):
        return "I_M"
    return ct


def family_palette(order: list) -> dict:
    """Build {channel_type: color} with family members sharing a hue.
    Within a family, lighter shade = earlier in `order` list."""
    from collections import defaultdict

    fam_groups = defaultdict(list)
    for ct in order:
        fam_groups[_family(ct)].append(ct)
    palette = {}
    for fam, types in fam_groups.items():
        cmap = _FAMILY_CMAP.get(fam, _cm.Greys)
        n = len(types)
        shades = np.linspace(0.4, 0.8, n) if n > 1 else [0.6]
        for ct, shade in zip(types, shades, strict=False):
            palette[ct] = cmap(shade)
    return palette


def order_types(
    df_in: pd.DataFrame,
    sort_by: str = "median",
    value_col: str = "neg_nle",
    min_count: int = 5,
) -> list:
    """Return channel-type labels (filtered to >= min_count) in the requested order."""
    counts = df_in["channel_type"].value_counts()
    keep = counts[counts >= min_count].index
    df_k = df_in[df_in["channel_type"].isin(keep)]
    stat = df_k.groupby("channel_type")[value_col]

    if sort_by == "median":
        return stat.median().sort_values().index.tolist()
    if sort_by == "min":
        return stat.min().sort_values().index.tolist()
    if sort_by == "family":
        med = stat.median()

        def _key(ct):
            fam = _family(ct)
            fi = (
                _FAMILY_ORDER.index(fam) if fam in _FAMILY_ORDER else len(_FAMILY_ORDER)
            )
            return (fi, float(med.get(ct, 9999)))

        return sorted(keep, key=_key)
    raise ValueError(f"Unknown sort_by='{sort_by}'. Use 'median', 'min', or 'family'.")


def _draw(df_in, x_col, y_col, order, ax, plot_type="violin", palette=None):
    """Draw a violin or box plot (seaborn, hue=x_col to avoid palette deprecation)."""
    if palette is None:
        palette = family_palette(order)
    kw = dict(
        data=df_in[df_in[x_col].isin(order)],
        x=x_col,
        y=y_col,
        order=order,
        hue=x_col,
        palette=palette,
        legend=False,
        ax=ax,
    )
    if plot_type == "violin":
        sns.violinplot(**kw, inner="box", density_norm="width")
    else:
        sns.boxplot(**kw)

---
## Analysis — neg_nle distribution per channel type

Full distribution of `neg_nle` values for each channel type (all particles, all seeds pooled).

Use `PLOT_TYPE`, `SORT_BY`, and `MIN_COUNT` in the config cell to change the appearance.

In [ ]:
# ── Plot configuration  (edit these flags) ────────────────────────────────────

# Plot type for distributions
PLOT_TYPE = "violin"  # "violin" | "box"

# Sort order for x-axis channel types
#   "median"  – ascending median neg_nle (lowest = best, leftmost)
#   "min"     – ascending minimum neg_nle (best single particle per type, leftmost)
#   "family"  – grouped by ion-channel family (base → I_A → I_M → I_NaP → unknown),
#               within each family sorted by median
SORT_BY = "family"  # "median" | "min" | "family"

# Minimum number of particles for a channel type to be included
MIN_COUNT = 8


# ── Distribution plot ───────────────────────────────────────────────────────────

with open(Path("panel_sizes_cm.yaml")) as _f:
    _panel_sizes = yaml.safe_load(_f)
_scale = 90 / 72
_fig_w = _panel_sizes["panel_analysis_a"]["width_cm"] / 2.54 * _scale
_fig_h = _panel_sizes["panel_analysis_a"]["height_cm"] / 2.54 * _scale

order = order_types(df_exp, sort_by=SORT_BY, value_col="neg_nle", min_count=MIN_COUNT)

with use_style("pyloric"):
    fig, ax = plt.subplots(figsize=(_fig_w, _fig_h))
    _draw(df_exp, "channel_type", "neg_nle", order, ax, plot_type=PLOT_TYPE)

    # Apply human-readable tick labels
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels([CHANNEL_LABELS.get(ct, ct) for ct in order])
    ax.set_xlabel("")
    ax.set_ylabel(r"$-\log\,p(x_o \mid m)$")

    # Force black spines and ticks (seaborn may override)
    for spine in ax.spines.values():
        spine.set_color("black")
    ax.tick_params(colors="black")
    ax.set_ylim(top=430, bottom=225)

    # Black horizontal grid lines
    # ax.yaxis.grid(False)
    ax.yaxis.grid(True, color="gray")

    plt.tight_layout()
    plt.savefig(
        FIGURES_DIR / f"distribution_{PLOT_TYPE}_{SORT_BY}.svg", bbox_inches="tight"
    )
    plt.show()

**ModelSMC consistently discovers biologically plausible ion-channel variants.**
Weight distributions across LLM-discovered ion-channel variants for the Allen HH task, over all 10 seeds. For each LLM-generated channel variant we show the distribution of SMC particle weights across all particles and seeds. Channel types are automatically identified using LLM calls. M-type ($I_M$) variants consistently achieve higher weights than the base model and A-type ($I_A$) channels. The figure shows that ModelSMC allows to focus on promising hypotheses (M-type) channels and to dismiss other hypotheses (e.g., $I_{NaP}$).
